In [2]:
import csv

DATA_FILE = "expenses.csv"
USERS = {"fatima": "fatima123", "rija": "rija123"}

In [10]:
def load_records():
    records = []
    try:
        with open(DATA_FILE, "r", newline="") as file:
            reader = csv.DictReader(file)
            for row in reader:
                records.append(row)
    except FileNotFoundError:
        print("No expense file yet. It will be created when you add data.")
    except PermissionError:
        print("Error: Cannot read the data file. Check file permissions.")
    finally:
        return records

In [4]:
def save_records(records):
    try:
        with open(DATA_FILE, "w", newline="") as file:
            fieldnames = ["expense_id", "description", "paid_by", "amount", "shared_with"]
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            for record in records:
                writer.writerow(record)
        print("Records saved successfully.")
    except PermissionError:
        print("Error: Cannot write to the data file. Check file permissions.")
    except Exception as e:
        print(f"Error saving records: {e}")


In [ ]:
def user_login():
    print("--- Group Expense Splitter Login ---")
    attempts = 0
    while attempts < 3:
        username = input("Username: ").strip()
        password = input("Password: ").strip()
        if username in USERS and USERS[username] == password:
            print(f"Welcome, {username}!")
            return True
        else:
            attempts += 1
            remaining = 3 - attempts
            if remaining > 0:
                print(f"Invalid credentials. {remaining} attempt(s) left.")
            else:
                print("Too many failed attempts. Access denied.")
    return False

In [5]:
def add_record(records):
    print("--- Add Expense ---")
    expense_id = input("Expense ID (e.g. E001): ").strip()

    for record in records:
        if record["expense_id"] == expense_id:
            print("An expense with that ID already exists.")
            return records

    description = input("Description: ").strip()
    paid_by = input("Paid by (name): ").strip()

    try:
        amount = float(input("Amount: ").strip())
        if amount <= 0:
            print("Amount must be greater than 0.")
            return records
    except ValueError:
        print("Invalid input. Amount must be a number.")
        return records

    shared_with = input("Shared with (comma-separated names, include payer): ").strip()
    if not shared_with:
        print("At least one person must share the expense.")
        return records

    new_record = {
        "expense_id": expense_id,
        "description": description,
        "paid_by": paid_by,
        "amount": str(amount),
        "shared_with": shared_with,
    }
    records.append(new_record)
    save_records(records)
    print(f"Expense '{description}' added successfully.")
    return records


In [6]:
def search_record(records):
    print("--- Search Expense ---")
    if not records:
        print("No records available to search.")
        return

    query = input("Enter Expense ID, description, or payer name: ").strip().lower()
    found = False

    for record in records:
        if (
            record["expense_id"].lower() == query
            or query in record["description"].lower()
            or query in record["paid_by"].lower()
        ):
            print(
                f"ID: {record['expense_id']} | "
                f"{record['description']} | "
                f"Paid by: {record['paid_by']} | "
                f"Amount: {record['amount']} | "
                f"Shared: {record['shared_with']}"
            )
            found = True

    if not found:
        print("No matching expense found.")

In [7]:
def generate_report(records):
    print("--- Expense Split Report ---")
    if not records:
        print("No records available to generate a report.")
        return {}

    balances = {}
    total_spent = 0.0

    for record in records:
        try:
            amount = float(record["amount"])
        except ValueError:
            print(f"Skipping invalid amount for {record['expense_id']}.")
            continue

        members = [m.strip() for m in record["shared_with"].split(",") if m.strip()]
        if not members:
            continue

        share = amount / len(members)
        payer = record["paid_by"].strip()
        total_spent += amount

        if payer not in balances:
            balances[payer] = 0.0
        balances[payer] += amount

        for member in members:
            if member not in balances:
                balances[member] = 0.0
            balances[member] -= share

    print(f"Total group spending: {round(total_spent, 2)}")
    print("Balances (positive = should receive, negative = should pay):")
    for name in balances:
        print(f"  {name}: {round(balances[name], 2)}")

    return balances

In [8]:
def show_menu():
    print("========== GROUP EXPENSE SPLITTER ==========")
    print("1. Add Expense")
    print("2. Search Expense")
    print("3. View All Expenses")
    print("4. Generate Split Report")
    print("5. Exit")
    print("============================================")


In [11]:
## main program
if user_login():
    records = load_records()

    while True:
        show_menu()
        try:
            choice = input("Enter your choice (1-5): ").strip()

            if choice == "1":
                records = add_record(records)
            elif choice == "2":
                search_record(records)
            elif choice == "3":
                view_all_records(records)
            elif choice == "4":
                generate_report(records)
            elif choice == "5":
                save_records(records)
                print("Goodbye!")
                break
            else:
                print("Invalid choice. Please enter a number between 1 and 5.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
else:
    print("Exiting program.")

--- Group Expense Splitter Login ---


Username:  admin
Password:  admin123


Welcome, admin!
========== GROUP EXPENSE SPLITTER ==========
1. Add Expense
2. Search Expense
3. View All Expenses
4. Generate Split Report
5. Exit


Enter your choice (1-5):  1


--- Add Expense ---


Expense ID (e.g. E001):  50
Description:  lunch and coffee
Paid by (name):  fatima
Amount:  45
Shared with (comma-separated names, include payer):  rija,zainab


... (3 outputs left out)

Enter your choice (1-5):  4


--- Expense Split Report ---
Total group spending: 545.0
Balances (positive = should receive, negative = should pay):
  zainab: 310.83
  fatima: -121.67
  ali: -166.67
  rija: -22.5
========== GROUP EXPENSE SPLITTER ==========
1. Add Expense
2. Search Expense
3. View All Expenses
4. Generate Split Report
5. Exit


Enter your choice (1-5):  5


Records saved successfully.
Goodbye!
